### Vector stores and retrievers 
This video tutorial will familiarize you with LangChain's vector store and retriever abstraction. These abstractions are designed to support retrieval of data-- from(vector) databse and other sources --for intergration with LLM workflows. They are important for application that fetch data to reasoned over as part of the model interface , as in the case of retrieveal - augumented generation.
We will cover
<li>Documents</li>
<li>Vector Stores</li>
<li>Retrievers</li>

### - Documents
Langchain implements a document abstraction , which is intended to represent a unit of text and associated metadata. It has two attributes:

- Page_content : a string representing the content:
- metadata: a dict containing arbitrary metadata.
to other documents , and other information . Note that an individual document object often represents a chunk of a large documents.

Let's generate some simple documents:


In [1]:
from langchain_core.documents import Document

documents = [
    Document(
        page_content="Dogs are great companiins , Know for their loyalty and friendliness.",
        metadata={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content="Cats aree independent pets that often enjoy their own space.",
        metadata={"source":"mammal-pets-doc"}
    ),
    Document(
        page_content="Golfish are popular pets for beginners, requiring relatively simple care.",
        metadata={"source":"fish-pets-doc"}
    ),
    Document(
        page_content="Parrots are intelligent birds capable of mimicking human speech.",
        metadata={"source":"bird-pets-doc"}
    ),
]

In [2]:
documents

[Document(metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companiins , Know for their loyalty and friendliness.'),
 Document(metadata={'source': 'mammal-pets-doc'}, page_content='Cats aree independent pets that often enjoy their own space.'),
 Document(metadata={'source': 'fish-pets-doc'}, page_content='Golfish are popular pets for beginners, requiring relatively simple care.'),
 Document(metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq
groq_api_key = os.getenv("GROQ_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

llm = ChatGroq(groq_api_key=groq_api_key , model="Qwen/Qwen3.6-27B")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '0.3.27'}}, output_version=None, client=<groq.resources.chat.completions.Completions object at 0x00000149164A3FA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000149164CC520>, model_name='Qwen/Qwen3.6-27B', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [5]:
%pip install langchain_huggingface

Note: you may need to restart the kernel to use updated packages.


In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddingss = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [5]:
## VectorStore
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(documents,embedding=embeddingss)
vectorstore



In [6]:
vectorstore.similarity_search("cat")

[Document(id='f333f5a3-29b5-4148-8d18-e8b4fe621298', metadata={'source': 'mammal-pets-doc'}, page_content='Cats aree independent pets that often enjoy their own space.'),
 Document(id='9186ea08-3521-453f-aec8-2a4fac1f3b2c', metadata={'source': 'fish-pets-doc'}, page_content='Golfish are popular pets for beginners, requiring relatively simple care.'),
 Document(id='635d7340-85cf-4e79-8475-4e5d38b0c441', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companiins , Know for their loyalty and friendliness.'),
 Document(id='85014a13-ecca-4ec2-ac9c-e28965acb4c7', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [7]:
await vectorstore.asimilarity_search('cat')

[Document(id='f333f5a3-29b5-4148-8d18-e8b4fe621298', metadata={'source': 'mammal-pets-doc'}, page_content='Cats aree independent pets that often enjoy their own space.'),
 Document(id='9186ea08-3521-453f-aec8-2a4fac1f3b2c', metadata={'source': 'fish-pets-doc'}, page_content='Golfish are popular pets for beginners, requiring relatively simple care.'),
 Document(id='635d7340-85cf-4e79-8475-4e5d38b0c441', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companiins , Know for their loyalty and friendliness.'),
 Document(id='85014a13-ecca-4ec2-ac9c-e28965acb4c7', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.')]

In [8]:
vectorstore.similarity_search_with_score("cat")

[(Document(id='f333f5a3-29b5-4148-8d18-e8b4fe621298', metadata={'source': 'mammal-pets-doc'}, page_content='Cats aree independent pets that often enjoy their own space.'),
  0.942659854888916),
 (Document(id='9186ea08-3521-453f-aec8-2a4fac1f3b2c', metadata={'source': 'fish-pets-doc'}, page_content='Golfish are popular pets for beginners, requiring relatively simple care.'),
  1.4901622533798218),
 (Document(id='635d7340-85cf-4e79-8475-4e5d38b0c441', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companiins , Know for their loyalty and friendliness.'),
  1.5762659311294556),
 (Document(id='85014a13-ecca-4ec2-ac9c-e28965acb4c7', metadata={'source': 'bird-pets-doc'}, page_content='Parrots are intelligent birds capable of mimicking human speech.'),
  1.66579270362854)]

### Retrivers 
Langchain VectorStore Object do noy subclass Runable, and so cannot immediately be intergrated into Langchain Expression Language Chains.

Langchain Retrievers are Runables , so they implement a stranard set of methods(e.g synchronous and asynchromouns invokeand batch Operations) and are designed to be incorporated in LCEL chains.

We cab create a simple version of this ourselves, without subclassing retriever. if we chosee what method wish to use to retrieve documents , we can create a runable easily below we will build one around the simliarity_serch method :

In [9]:
from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retriever = RunnableLambda(vectorstore.similarity_search).bind(k=1)
retriever.batch(["cat" , "dog"])

[[Document(id='f333f5a3-29b5-4148-8d18-e8b4fe621298', metadata={'source': 'mammal-pets-doc'}, page_content='Cats aree independent pets that often enjoy their own space.')],
 [Document(id='635d7340-85cf-4e79-8475-4e5d38b0c441', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companiins , Know for their loyalty and friendliness.')]]

Vectorstores implemation an as_retriever method that will generate a Retriever, sepecifically a VectorStoreRetriever. These retrievers includes specific search_type and search_kwargs attributes that identify what method of the undelying vector store to call , nad how to parameterize them. Fro instance we ca replicates the above with of the following:


In [10]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1}
)
retriever.batch(["cat","dog"])

[[Document(id='f333f5a3-29b5-4148-8d18-e8b4fe621298', metadata={'source': 'mammal-pets-doc'}, page_content='Cats aree independent pets that often enjoy their own space.')],
 [Document(id='635d7340-85cf-4e79-8475-4e5d38b0c441', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companiins , Know for their loyalty and friendliness.')]]

In [ ]:
## RAG

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
message ="""
Answer this question using the provided context only.
{question}
Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([("human",message)])

rag_chain ={"context":retriever , "question":RunnablePassthrough()}|prompt|llm

response = rag_chain.invoke("tell me about dogs")
print(response.content)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Question:** "tell me about dogs"
   - **Constraint:** "Answer this question using the provided context only."
   - **Context:** `[Document(id='635d7340-85cf-4e79-8475-4e5d38b0c441', metadata={'source': 'mammal-pets-doc'}, page_content='Dogs are great companiins , Know for their loyalty and friendliness.')]`

2.  **Extract Key Information from Context:**
   - The context states: "Dogs are great companiins , Know for their loyalty and friendliness." (Note: there are typos in the context "companiins" and capitalization "Know", but I should convey the meaning accurately based on the text).

3.  **Formulate Answer based *only* on Context:**
   - I need to stick strictly to the provided text.
   - Draft: Based on the provided context, dogs are great companions and are known for their loyalty and friendliness.
   - Check against constraint: Does it use only the provided context? Yes. Does it answer the question? Yes.

4. 